# Baseline B — Language-specific topological (fastText + PH)

Computes Vietoris–Rips persistent homology barcodes on the cosine distance
matrices built from language-specific fastText CC-300 vectors, then extracts
the Kushnareva-style feature vector for each (language, domain) combination.

**Sub-issue:** `ph-project-ssa.5`

**Output artifacts:**
- `results/baseline_B/barcodes/<lang>_<domain>.npz` — raw barcode arrays
- `results/baseline_B/barcodes/<domain>.png` — H0 and H1 barcode plots (3-language comparison)
- `results/baseline_B/features.csv` — flat feature table (9 rows × 24 feature columns)

In [1]:
# Ensure the repo root (parent of notebooks/) is on sys.path so that
# `import baselines` resolves correctly regardless of how the kernel was
# launched or what the CWD is.
import sys
import pathlib

_NOTEBOOK_DIR = pathlib.Path(__file__).resolve().parent if '__file__' in dir() else pathlib.Path('.').resolve()
_REPO_ROOT = _NOTEBOOK_DIR.parent if _NOTEBOOK_DIR.name == 'notebooks' else _NOTEBOOK_DIR

if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

print(f'Repo root: {_REPO_ROOT}')
print(f'sys.path[0]: {sys.path[0]}')

Repo root: ~/ph-project
sys.path[0]: ~/ph-project


In [2]:
import warnings

import matplotlib
matplotlib.use('Agg')  # non-interactive backend for headless execution
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

from baselines import SUPPORTED_LANGUAGES
from baselines.vectors import load_fasttext
from baselines.distances import extract_term_vectors, cosine_distance_matrix
from baselines.topology import rips_barcode, barcode_features

# Suppress gensim/numpy deprecation noise during loading
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

REPO_ROOT = _REPO_ROOT
CANON_ROOT = REPO_ROOT / 'canon-terms'
RESULTS_ROOT = REPO_ROOT / 'results' / 'baseline_B'

(RESULTS_ROOT / 'barcodes').mkdir(parents=True, exist_ok=True)

# Domains and languages — same constants as baseline A
DOMAINS = ['color', 'emotion', 'kinship']
LANGUAGES = sorted(SUPPORTED_LANGUAGES)  # deterministic order: ['en', 'es', 'ru']

print(f'Languages: {LANGUAGES}')
print(f'Domains:   {DOMAINS}')
print(f'Results:   {RESULTS_ROOT}')

Languages: ['en', 'es', 'ru']
Domains:   ['color', 'emotion', 'kinship']
Results:   ~/ph-project/results/baseline_B


## Step 1 — Load canon terms and fastText CC-300 vectors

Read all 9 (language, domain) YAML files, then load each language's CC-300 model
once and reuse it across domains.  Mirrors baseline A exactly.

In [3]:
def load_canon_terms(lang: str, domain: str) -> list:
    """Read canon term strings from canon-terms/<lang>/<domain>.yaml."""
    path = CANON_ROOT / lang / f'{domain}.yaml'
    with open(path) as f:
        data = yaml.safe_load(f)
    return [entry['term'] for entry in data['terms']]


# Pre-load all term lists
terms_by_lang_domain = {}
for lang in LANGUAGES:
    for domain in DOMAINS:
        terms = load_canon_terms(lang, domain)
        terms_by_lang_domain[(lang, domain)] = terms
        print(f'{lang}/{domain}: {len(terms)} terms')

en/color: 11 terms
en/emotion: 18 terms
en/kinship: 27 terms
es/color: 11 terms
es/emotion: 22 terms
es/kinship: 32 terms
ru/color: 12 terms
ru/emotion: 19 terms
ru/kinship: 34 terms


In [4]:
vectors_by_lang = {}
for lang in LANGUAGES:
    print(f'Loading fastText CC-300 for lang={lang!r}...')
    vectors_by_lang[lang] = load_fasttext(lang, 'cc')
    print(f'  dim={vectors_by_lang[lang].vector_size}, loaded.')

print('\nAll vectors loaded.')

Loading fastText CC-300 for lang='en'...


  dim=300, loaded.
Loading fastText CC-300 for lang='es'...


  dim=300, loaded.
Loading fastText CC-300 for lang='ru'...


  dim=300, loaded.

All vectors loaded.


## Step 2 — Extract term vectors and compute cosine distance matrices

For each (language, domain), extract a `(n_found, 300)` matrix using
`extract_term_vectors` with `strategy='head'`, then compute the pairwise cosine
distance matrix.  This is identical to the baseline A pipeline.

In [5]:
matrices = {}
found_terms = {}
distance_matrices = {}

for lang in LANGUAGES:
    kv = vectors_by_lang[lang]
    for domain in DOMAINS:
        terms = terms_by_lang_domain[(lang, domain)]
        matrix, mask = extract_term_vectors(
            terms, kv, strategy='head', lang=lang, domain=domain
        )
        ft = [t for t, m in zip(terms, mask) if m]
        matrices[(lang, domain)] = matrix
        found_terms[(lang, domain)] = ft

        D = cosine_distance_matrix(matrix)
        distance_matrices[(lang, domain)] = D
        oov = int((~mask).sum())
        status = f'  [OOV: {oov} terms]' if oov else ''
        print(f'{lang}/{domain}: {matrix.shape[0]}/{len(terms)} found, D shape={D.shape}, '
              f'range=[{D.min():.4f}, {D.max():.4f}]{status}')

print('\nDistance matrices computed.')

en/color: 11/11 found, D shape=(11, 11), range=[0.0000, 0.4983]
en/emotion: 18/18 found, D shape=(18, 18), range=[0.0000, 0.8245]
en/kinship: 27/27 found, D shape=(27, 27), range=[0.0000, 0.5951]
es/color: 11/11 found, D shape=(11, 11), range=[0.0000, 0.5071]
es/emotion: 22/22 found, D shape=(22, 22), range=[0.0000, 1.0033]
es/kinship: 32/32 found, D shape=(32, 32), range=[0.0000, 0.8513]
ru/color: 12/12 found, D shape=(12, 12), range=[0.0000, 0.4009]
ru/emotion: 19/19 found, D shape=(19, 19), range=[0.0000, 0.8991]
ru/kinship: 34/34 found, D shape=(34, 34), range=[0.0000, 0.7324]

Distance matrices computed.


## Step 3 — Vietoris–Rips barcodes

Run Ripser (CPU) on each cosine distance matrix to compute H0 and H1 barcodes.
The single infinite H0 bar is stripped automatically by `rips_barcode`.

Also save raw barcode arrays to `results/baseline_B/barcodes/<lang>_<domain>.npz`.

In [6]:
barcodes = {}

for lang in LANGUAGES:
    for domain in DOMAINS:
        D = distance_matrices[(lang, domain)]
        bc = rips_barcode(D, max_dim=1)
        barcodes[(lang, domain)] = bc

        n_h0 = len(bc[0])
        n_h1 = len(bc[1])
        print(f'{lang}/{domain}: H0 finite bars={n_h0}, H1 finite bars={n_h1}')

        # Save raw barcode arrays as .npz
        npz_path = RESULTS_ROOT / 'barcodes' / f'{lang}_{domain}.npz'
        np.savez(
            npz_path,
            h0_birth=bc[0]['birth'], h0_death=bc[0]['death'],
            h1_birth=bc[1]['birth'], h1_death=bc[1]['death'],
        )

print('\nBarcodes computed and saved.')

en/color: H0 finite bars=10, H1 finite bars=0
en/emotion: H0 finite bars=17, H1 finite bars=7
en/kinship: H0 finite bars=26, H1 finite bars=19
es/color: H0 finite bars=10, H1 finite bars=1
es/emotion: H0 finite bars=21, H1 finite bars=6
es/kinship: H0 finite bars=31, H1 finite bars=22
ru/color: H0 finite bars=11, H1 finite bars=2
ru/emotion: H0 finite bars=18, H1 finite bars=6
ru/kinship: H0 finite bars=31, H1 finite bars=18

Barcodes computed and saved.


## Step 4 — Barcode plots (H0 and H1, per domain across languages)

Each domain produces one figure with a 3-row × 2-column layout:
- rows: en, es, ru
- columns: H0 (connected components), H1 (loops)

Saved to `results/baseline_B/barcodes/<domain>.png`.

In [7]:
def plot_barcode(ax, bc_dim, title, color='steelblue'):
    """Draw a barcode diagram on ax.  Each bar is a horizontal line [birth, death]."""
    n = len(bc_dim)
    if n == 0:
        ax.set_title(title, fontsize=9)
        ax.text(0.5, 0.5, 'empty', ha='center', va='center',
                transform=ax.transAxes, fontsize=8, color='grey')
        ax.set_yticks([])
        return
    # Sort by persistence (death - birth), longest first
    lengths = bc_dim['death'] - bc_dim['birth']
    order = np.argsort(lengths)[::-1]
    for i, idx in enumerate(order):
        ax.plot([bc_dim['birth'][idx], bc_dim['death'][idx]], [i, i],
                color=color, linewidth=2.0, solid_capstyle='butt')
    ax.set_title(title, fontsize=9)
    ax.set_ylabel('bar index', fontsize=7)
    ax.set_xlabel('filtration value', fontsize=7)
    ax.tick_params(labelsize=7)


for domain in DOMAINS:
    fig, axes = plt.subplots(nrows=3, ncols=2,
                             figsize=(10, 7),
                             constrained_layout=True)
    fig.suptitle(
        f'Barcodes — {domain.capitalize()} domain\n'
        f'(fastText CC-300, cosine distance, Vietoris–Rips)',
        fontsize=11
    )

    colors_h0 = {'en': 'steelblue', 'es': 'darkorange', 'ru': 'seagreen'}
    colors_h1 = {'en': 'royalblue', 'es': 'orangered', 'ru': 'mediumseagreen'}

    for row_idx, lang in enumerate(LANGUAGES):
        bc = barcodes[(lang, domain)]
        plot_barcode(axes[row_idx, 0], bc[0],
                     title=f'{lang.upper()} — H0 (components)',
                     color=colors_h0[lang])
        plot_barcode(axes[row_idx, 1], bc[1],
                     title=f'{lang.upper()} — H1 (loops)',
                     color=colors_h1[lang])

    out_path = RESULTS_ROOT / 'barcodes' / f'{domain}.png'
    fig.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved: {out_path.name}')

print('\nAll barcode plots saved.')

Saved: color.png


Saved: emotion.png


Saved: kinship.png

All barcode plots saved.


## Step 5 — Kushnareva-style barcode features

Extract the full 24-feature vector (12 per homology dimension) for each
(language, domain) using `barcode_features`.  Feature names mirror
`reference/ripser_count.py` exactly, including the `h{d}_v` = std convention.

In [8]:
features_by_lang_domain = {}

for lang in LANGUAGES:
    for domain in DOMAINS:
        bc = barcodes[(lang, domain)]
        feats = barcode_features(bc)
        features_by_lang_domain[(lang, domain)] = feats

# Quick sanity check — print feature count and first row
sample_lang, sample_domain = LANGUAGES[0], DOMAINS[0]
sample_feats = features_by_lang_domain[(sample_lang, sample_domain)]
print(f'Feature count: {len(sample_feats)}')
print(f'Sample features ({sample_lang}/{sample_domain}):')
for k, v in sample_feats.items():
    print(f'  {k}: {v:.6f}')

Feature count: 24
Sample features (en/color):
  h0_s: 2.402799
  h0_m: 0.240280
  h0_v: 0.070160
  h0_e: 2.259548
  h0_n_d_m_t0.25: 4.000000
  h0_n_d_m_t0.5: 0.000000
  h0_n_d_m_t0.75: 0.000000
  h0_n_b_l_t0.25: 10.000000
  h0_n_b_l_t0.5: 10.000000
  h0_n_b_l_t0.75: 10.000000
  h0_t_b: 0.000000
  h0_t_d: 0.353995
  h1_s: 0.000000
  h1_m: 0.000000
  h1_v: 0.000000
  h1_e: 0.000000
  h1_n_d_m_t0.25: 0.000000
  h1_n_d_m_t0.5: 0.000000
  h1_n_d_m_t0.75: 0.000000
  h1_n_b_l_t0.25: 0.000000
  h1_n_b_l_t0.5: 0.000000
  h1_n_b_l_t0.75: 0.000000
  h1_t_b: 0.000000
  h1_t_d: 0.000000


## Step 6 — Assemble and save feature table

Assemble a flat CSV with one row per (language, domain) and one column per
feature, then save to `results/baseline_B/features.csv`.

In [9]:
rows = []
for lang in LANGUAGES:
    for domain in DOMAINS:
        row = {'language': lang, 'domain': domain}
        row.update(features_by_lang_domain[(lang, domain)])
        rows.append(row)

df = pd.DataFrame(rows)
csv_path = RESULTS_ROOT / 'features.csv'
df.to_csv(csv_path, index=False)
print(f'Saved: {csv_path}')
df

Saved: ~/ph-project/results/baseline_B/features.csv


,language,domain,h0_s,h0_m,h0_v,h0_e,h0_n_d_m_t0.25,h0_n_d_m_t0.5,h0_n_d_m_t0.75,h0_n_b_l_t0.25,...,h1_v,h1_e,h1_n_d_m_t0.25,h1_n_d_m_t0.5,h1_n_d_m_t0.75,h1_n_b_l_t0.25,h1_n_b_l_t0.5,h1_n_b_l_t0.75,h1_t_b,h1_t_d
0,en,color,2.402799,0.240280,0.070160,2.259548,4.0,0.0,0.0,10.0,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000
1,en,emotion,7.078964,0.416410,0.081801,2.813974,17.0,2.0,0.0,17.0,...,0.022653,1.522815,7.0,7.0,0.0,0.0,0.0,7.0,0.547855,0.616293
2,en,kinship,4.220058,0.162310,0.028753,3.242051,0.0,0.0,0.0,26.0,...,0.030548,2.776006,11.0,0.0,0.0,18.0,19.0,19.0,0.144890,0.254304
3,es,color,2.346132,0.234613,0.053828,2.274858,6.0,0.0,0.0,10.0,...,0.000000,0.000000,1.0,0.0,0.0,0.0,1.0,1.0,0.303743,0.312558
4,es,emotion,8.336538,0.396978,0.108744,3.005472,19.0,1.0,0.0,21.0,...,0.028773,1.608096,6.0,3.0,0.0,0.0,6.0,6.0,0.487190,0.586815
5,es,kinship,6.713316,0.216559,0.074165,3.382942,6.0,0.0,0.0,31.0,...,0.037938,2.740691,20.0,0.0,0.0,10.0,22.0,22.0,0.190071,0.323913
6,ru,color,1.727206,0.157019,0.030540,2.379182,0.0,0.0,0.0,11.0,...,0.004999,0.650616,1.0,0.0,0.0,2.0,2.0,2.0,0.243137,0.265398
7,ru,emotion,7.267117,0.403729,0.129055,2.838898,17.0,3.0,0.0,18.0,...,0.025904,1.567193,6.0,4.0,0.0,0.0,5.0,6.0,0.475854,0.563020
8,ru,kinship,7.777062,0.250873,0.058365,3.407583,14.0,0.0,0.0,31.0,...,0.032139,2.622793,18.0,0.0,0.0,1.0,18.0,18.0,0.274595,0.380902


## Step 7 — Artifact check

In [10]:
# Verify all expected artifacts exist
expected_npz = [RESULTS_ROOT / 'barcodes' / f'{l}_{d}.npz'
                for l in LANGUAGES for d in DOMAINS]
expected_png = [RESULTS_ROOT / 'barcodes' / f'{d}.png' for d in DOMAINS]
expected_csv = RESULTS_ROOT / 'features.csv'

missing = []
for p in expected_npz + expected_png + [expected_csv]:
    if not p.exists():
        missing.append(str(p))

if missing:
    print(f'MISSING ARTIFACTS ({len(missing)}):')
    for m in missing:
        print(f'  {m}')
else:
    print('All artifacts present:')
    print(f'  {len(expected_npz)} barcode .npz files')
    print(f'  {len(expected_png)} barcode .png plots')
    print(f'  features.csv')

assert not missing, f'Missing artifacts: {missing}'

All artifacts present:
  9 barcode .npz files
  3 barcode .png plots
  features.csv


## Discussion — Early look at prediction P1

**Prediction P1:** Russian color terms should show more persistent H0 structure
than English or Spanish, because Russian lexicalizes the blue/light-blue
distinction as two basic color terms (синий / голубой) that may form a distinct
cluster in the embedding space — appearing as an extra, persistent H0 bar in
the barcode.

We compare `h0_s` (total persistence) and `h0_e` (persistence entropy) for the
color domain across EN, ES, and RU.

In [11]:
print('Prediction P1 — Color domain H0 comparison')
print('=' * 50)
print(f'{"lang":<6}  {"h0_s":>10}  {"h0_e":>10}  {"h0_m":>10}  {"h0_v":>10}  {"n_finite_h0":>12}')
for lang in LANGUAGES:
    feats = features_by_lang_domain[(lang, 'color')]
    bc = barcodes[(lang, 'color')]
    n_h0 = len(bc[0])
    print(f'{lang:<6}  {feats["h0_s"]:>10.4f}  {feats["h0_e"]:>10.4f}  '
          f'{feats["h0_m"]:>10.4f}  {feats["h0_v"]:>10.4f}  {n_h0:>12d}')

Prediction P1 — Color domain H0 comparison
lang          h0_s        h0_e        h0_m        h0_v   n_finite_h0
en          2.4028      2.2595      0.2403      0.0702            10
es          2.3461      2.2749      0.2346      0.0538            10
ru          1.7272      2.3792      0.1570      0.0305            11


## Qualitative interpretation of P1

The table above shows the H0 persistence summary statistics for the color domain
in each language.  A higher `h0_s` (total persistence) means the point cloud
has more spread-out component structure — components merge later in the filtration,
implying more distinct clusters.  A higher `h0_e` (persistence entropy) means
the spread is more uniform across multiple components, rather than one dominant
gap.

At this stage the comparison is exploratory: with only 11–12 color terms per
language and a single static embedding model, the signal is weak and highly
sensitive to the specific fastText training corpus.  The main hypothesis test
(P1) will be re-examined with the mBERT attention-based topology in Phase 3,
where the contextual signal should be stronger.  What this baseline provides is
a reference point — if the static fastText embedding already shows elevated RU
color persistence relative to EN/ES, that would suggest the distributional
signal is robust across embedding types.  If not, it is still consistent with P1
holding only at the contextual (mBERT) level.